# Cell-type Deconvolution

##### Franziska Niemeyer, 2026-03-18

In [ ]:
import sys
!{sys.executable} -m pip install --timeout 300 --retries 10 torch cell2location

In [ ]:
import squidpy as sq
import scanpy as sc
import torch
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# os.environ["THEANO_FLAGS"] = 'device=cuda,floatX=float32,force_device=True'
import cell2location as c2l

In [ ]:
WORKING_DIR = "."
DATA_DIR = "data"
ADATA = "../quality_control/primary-cohort/adata.h5ad"
OUT_DIR = os.path.join(WORKING_DIR, "spot-deconvolution_zheng")
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

sc.settings.figdir = OUT_DIR

In [ ]:
results_folder = OUT_DIR

ref_run_name = f'{results_folder}/reference_signatures'
run_name = f'{results_folder}/cell2location_map'

In [ ]:
adata_sc = sc.read_h5ad(f"{DATA_DIR}/OvC_adata_zheng_preprocessed.h5ad")
adata_sc

In [ ]:
adata_sc.X = adata_sc.X.tocsr().astype('float32')

In [ ]:
adata_st = sc.read_h5ad(ADATA)

In [ ]:
shared_features = [feature for feature in adata_st.var_names if feature in adata_sc.var_names]

# filter
adata_sc = adata_sc[:, adata_sc.var.index.isin(shared_features)].copy()
adata_st = adata_st[:, shared_features].copy() # probably no need to filter the spatial gene set
# reorder genes to have them match between spatial data and sc
adata_sc = adata_sc[:, shared_features].copy()

print(f'Number of shared genes between spatial data and reference sc: {adata_st.shape[1]}')

In [ ]:
selected = c2l.utils.filtering.filter_genes(
    adata_sc, cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12
)

adata_sc = adata_sc[:, selected].copy()
adata_st = adata_st[:, selected].copy()

In [ ]:
adata_sc.obs['cell_type'].value_counts()

In [ ]:
print(f"The spatial AnnData object has {adata_st.shape[0]} obs and {adata_st.shape[1]} features.")

In [ ]:
print(f"The single-cell AnnData object has {adata_sc.shape[0]} obs and {adata_sc.shape[1]} features.")

In [ ]:
adata_sc.obs

In [ ]:
c2l.models.RegressionModel.setup_anndata(
    adata=adata_sc,
    batch_key='Patients',
    labels_key="cell_type",
)

In [ ]:
model = c2l.models.RegressionModel(adata_sc)
model.view_anndata_setup()

In [ ]:
model.train(max_epochs=250, batch_size=2500, train_size=1, lr=0.002)

In [ ]:
# Save model
model.save(ref_run_name, overwrite=True)

In [ ]:
model = c2l.models.RegressionModel.load(f"{ref_run_name}", adata_sc)

In [ ]:
import matplotlib.pyplot as plt
from aquarel import load_theme
import cmcrameri

theme = (
    load_theme("umbra_light").set_overrides({
    "figure.facecolor": 'white',
    "axes.facecolor": 'white'
})
    .set_grid(draw=False)
)

with theme:
    fig = plt.figure(figsize=(4,3))
    model.plot_history(20)
    plt.title("ELBO loss reference signature")
    plt.savefig(f"{ref_run_name}/ELBO_loss.pdf", bbox_inches='tight')
    plt.show()

In [ ]:
# export the estimated cell abundance (summary of the posterior distribution).
adata_sc = model.export_posterior(
    adata_sc, sample_kwargs={'num_samples': 1000, 'batch_size': 2500}
)

In [ ]:
model.plot_QC()

In [ ]:
adata_sc = model.export_posterior(
    adata_sc, sample_kwargs={'num_samples': 1000, 'batch_size': model.adata.n_obs}
)

In [ ]:
# Save anndata object with results
adata_file = f"{ref_run_name}/adata_sc.h5ad"
adata_sc.write(adata_file)
adata_file

In [ ]:
adata_file = f"{ref_run_name}/adata_st.h5ad"
adata_st.write(adata_file)
adata_file